In [1]:
import csv
import time
import struct
import threading
from dataclasses import dataclass, asdict
from pathlib import Path
from queue import Queue, Empty

import serial

# Reuse your uploaded protocol helpers
from protocol import (
    build_frame,
    PROTOCOL_VER,
    FLAG_DATA,
    ID_PC,
    ID_DRONE,
)

In [ ]:
@dataclass
class TestConfig:
    # Serial ports
    arduino_port: str = "COM6"      # change
    arduino_baud: int = 115200

    drone_port: str = "COM7"        # change
    drone_baud: int = 115200

    # EDF sweep
    start_power: int = 0
    end_power: int = 100
    power_step: int = 5
    dwell_time_s: float = 3.0       # time spent at each power step

    # Servo angles (real angles in degrees)
    # Encoded value sent is angle + 90 (max 37deg)
    x_plus_angle_deg: int = 0
    x_minus_angle_deg: int = 0
    y_plus_angle_deg: int = 0
    y_minus_angle_deg: int = 0

    # Load-cell options
    use_second_loadcell: bool = True

    # Logging
    output_csv: str = "edf_test_log.csv"

    # Optional settling times
    pre_test_idle_s: float = 2.0
    post_test_idle_s: float = 2.0

cfg = TestConfig()
cfg

TestConfig(arduino_port='COM6', arduino_baud=115200, drone_port='COM7', drone_baud=115200, start_power=0, end_power=100, power_step=5, dwell_time_s=3.0, x_plus_angle_deg=0, x_minus_angle_deg=0, y_plus_angle_deg=0, y_minus_angle_deg=0, use_second_loadcell=True, output_csv='edf_test_log.csv', pre_test_idle_s=2.0, post_test_idle_s=2.0)

: 

In [4]:
OPT_CAL_PARAM = 0x60

def encode_servo_angle(angle_deg: int) -> int:
    """
    Drone expects stored_angle = real_angle + 90
    Example:
      -10 deg -> 80
        0 deg -> 90
       10 deg -> 100
    """
    encoded = angle_deg + 90
    if encoded < -128 or encoded > 127:
        raise ValueError(f"Encoded servo value out of int8 range: {encoded}")
    return encoded

def build_calib_frame(
    edf_pwr_percent: int,
    x_plus_angle_deg: int = 0,
    x_minus_angle_deg: int = 0,
    y_plus_angle_deg: int = 0,
    y_minus_angle_deg: int = 0,
) -> bytes:
    if not (0 <= edf_pwr_percent <= 100):
        raise ValueError("edf_pwr_percent must be in range 0..100")

    payload = struct.pack(
        "<bbbbb",
        edf_pwr_percent,
        encode_servo_angle(x_plus_angle_deg),
        encode_servo_angle(x_minus_angle_deg),
        encode_servo_angle(y_plus_angle_deg),
        encode_servo_angle(y_minus_angle_deg),
    )

    frame = build_frame(
        version=PROTOCOL_VER,
        flags=FLAG_DATA,
        src=ID_PC,
        dst=ID_DRONE,
        opcode=OPT_CAL_PARAM,
        payload=payload,
    )
    return frame

In [5]:
@dataclass
class SensorSample:
    pc_time_s: float
    arduino_time_us: int
    load1: float
    load2: float

class ArduinoReader(threading.Thread):
    def __init__(self, ser: serial.Serial, out_queue: Queue):
        super().__init__(daemon=True)
        self.ser = ser
        self.out_queue = out_queue
        self._stop = threading.Event()

    def stop(self):
        self._stop.set()

    def run(self):
        while not self._stop.is_set():
            try:
                line = self.ser.readline().decode("utf-8", errors="ignore").strip()
                if not line or line.startswith("#"):
                    continue

                parts = line.split(",")
                if len(parts) < 3:
                    continue

                arduino_time_us = int(parts[0])
                load1 = float(parts[1])
                load2 = float(parts[2])

                sample = SensorSample(
                    pc_time_s=time.time(),
                    arduino_time_us=arduino_time_us,
                    load1=load1,
                    load2=load2,
                )
                self.out_queue.put(sample)

            except Exception:
                # keep running even if one line is corrupted
                continue

In [6]:
def run_edf_test(cfg: TestConfig):

    power_values = list(range(cfg.start_power, cfg.end_power + 1, cfg.power_step))
    if power_values[-1] != cfg.end_power:
        power_values.append(cfg.end_power)

    output_path = Path(cfg.output_csv)

    sensor_queue = Queue()
    rows = []

    print("Opening serial ports...")

    arduino_ser = serial.Serial(cfg.arduino_port, cfg.arduino_baud, timeout=0.2)
    drone_ser = serial.Serial(cfg.drone_port, cfg.drone_baud, timeout=0.2)

    time.sleep(2)  # allow ports to stabilize

    reader = ArduinoReader(arduino_ser, sensor_queue)
    reader.start()

    current_power = 0
    last_step_change_pc_s = time.time()
    transition_flag = 0

    def drain_queue():
        nonlocal transition_flag
        while True:
            try:
                sample = sensor_queue.get_nowait()
            except Empty:
                break

            row = {
                "pc_time_s": sample.pc_time_s,
                "arduino_time_us": sample.arduino_time_us,
                "edf_power_percent": current_power,
                "load1": sample.load1,
                "load2": sample.load2 if cfg.use_second_loadcell else "",
                "transition_flag": transition_flag,
                "time_since_step_s": sample.pc_time_s - last_step_change_pc_s,
            }

            rows.append(row)

            transition_flag = 0

    try:

        # ---------------------------
        # TARE BEFORE TEST
        # ---------------------------

        print("Sending TARE command to Arduino...")

        arduino_ser.reset_input_buffer()
        arduino_ser.write(b"T\n")

        while True:
            line = arduino_ser.readline().decode(errors="ignore").strip()

            if line:
                print("Arduino:", line)

            if "TARE_DONE" in line:
                break

        print("Tare completed")

        print("Waiting for system to stabilize...")
        time.sleep(2)

        # ---------------------------
        # START POWER SWEEP
        # ---------------------------

        print("Starting EDF sweep")

        for pwr in power_values:

            frame = build_calib_frame(
                edf_pwr_percent=pwr,
                x_plus_angle_deg=cfg.x_plus_angle_deg,
                x_minus_angle_deg=cfg.x_minus_angle_deg,
                y_plus_angle_deg=cfg.y_plus_angle_deg,
                y_minus_angle_deg=cfg.y_minus_angle_deg,
            )

            drone_ser.write(frame)
            drone_ser.flush()

            current_power = pwr
            last_step_change_pc_s = time.time()
            transition_flag = 1

            print(f"Sent EDF power = {pwr}%")

            dwell_start = time.time()

            while time.time() - dwell_start < cfg.dwell_time_s:
                drain_queue()
                time.sleep(0.002)

        # ---------------------------
        # RETURN TO 0% POWER
        # ---------------------------

        print("Returning EDF to 0%")

        frame = build_calib_frame(
            edf_pwr_percent=0,
            x_plus_angle_deg=cfg.x_plus_angle_deg,
            x_minus_angle_deg=cfg.x_minus_angle_deg,
            y_plus_angle_deg=cfg.y_plus_angle_deg,
            y_minus_angle_deg=cfg.y_minus_angle_deg,
        )

        drone_ser.write(frame)
        drone_ser.flush()

        current_power = 0
        last_step_change_pc_s = time.time()
        transition_flag = 1

        time.sleep(2)

        drain_queue()

    finally:

        reader.stop()
        time.sleep(0.2)

        drain_queue()

        try:
            arduino_ser.close()
        except:
            pass

        try:
            drone_ser.close()
        except:
            pass

    # ---------------------------
    # SAVE CSV
    # ---------------------------

    fieldnames = [
        "pc_time_s",
        "arduino_time_us",
        "edf_power_percent",
        "load1",
        "load2",
        "transition_flag",
        "time_since_step_s",
    ]

    with open(output_path, "w", newline="") as f:

        writer = csv.DictWriter(f, fieldnames=fieldnames)

        writer.writeheader()
        writer.writerows(rows)

    print(f"Saved {len(rows)} rows to {output_path}")

    return rows

In [8]:
cfg = TestConfig(
    arduino_port="COM7",     # change
    drone_port="COM7",       # change
    start_power=0,
    end_power=100,
    power_step=10,
    dwell_time_s=3.0,
    output_csv="edf_motor_characterization.csv",
    use_second_loadcell=True,
)

rows = run_edf_test(cfg)

Opening serial ports...


SerialException: could not open port 'COM7': PermissionError(13, 'Access is denied.', None, 5)

In [ ]:
import pandas as pd

csv_file = "edf_motor_characterization.csv"

df = pd.read_csv(csv_file)

# ==========================
# CALIBRATION CONSTANTS
# ==========================

counts_per_kg_loadcell1 = 220000
counts_per_kg_loadcell2 = 220000

g = 9.80665

# ==========================
# Convert raw → Newtons
# ==========================

df["force1_N"] = (df["load1"] / counts_per_kg_loadcell1) * g

if "load2" in df.columns:
    df["force2_N"] = (df["load2"] / counts_per_kg_loadcell2) * g
else:
    df["force2_N"] = None

# optional total thrust
df["force_total_N"] = df["force1_N"] + df["force2_N"].fillna(0)

print(df.head())

# Save new CSV
out = "edf_motor_characterization_newtons.csv"
df.to_csv(out, index=False)

print("Saved converted data:", out)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(df["edf_power_percent"], df["force_total_N"], '.', alpha=0.4)

plt.xlabel("EDF Power (%)")
plt.ylabel("Thrust (N)")
plt.title("EDF Static Thrust Characteristic")
plt.grid()

plt.show()